# 09 — Fully nested ordinal ensemble

The authoritative algorithm lives in `src/experiment.py`. This notebook uses the same command-line workflow, not a separately implemented model. It evaluates generic inner-only hyperparameter selection, training-only early stopping, and QWK calibration on untouched outer folds. Historical best Optuna parameters are not imported.

Primary policy: **equal-weight CatBoost + XGBoost + ExtraTrees + Ridge**, followed by inner-OOF thresholds. The deployed model repeats that exact algorithm on all labeled training rows. A weighted ensemble remains a secondary comparison, never a silent replacement.

Project design followed earlier exploratory work on this dataset; resampling estimates are not a newly collected untouched cohort. Bootstrap intervals are conditional fixed-prediction diagnostics. No stable gain or clinical suitability is assumed.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.config import RESULTS_DIR, TARGET
from src.experiment import CANDIDATES, PRIMARY, ExperimentConfig, prepare_data, results_match, run_experiment
from src.verify import verify_artifacts

config = ExperimentConfig()
FORCE_RETRAIN = False
print("Reference configuration:", config)

## 1. Row-local preparation and labeled-cohort audit

Read raw CSVs and regenerate Layer A. The experiment never trusts a stale precomputed parquet. All measurement-missing flags exclude season metadata; they make no claim about assessment administration. Median is the actual preprocessing function default. Learned statistics remain training-partition-only.

In [ ]:
data = prepare_data()
print("Raw/labeled/unlabeled/test counts:",
      data["provenance"]["raw_train_rows"], len(data["y"]),
      data["provenance"]["unlabeled_rows"], len(data["X_test"]))
print("Feature count:", len(data["feature_columns"]))
print("Class counts:", data["provenance"]["class_counts"])
print("Cleaning audit:", data["provenance"]["cleaning"])

## 2. Explicit candidate grids and controlled baseline

Candidate 0 defines the untuned reference for each member. Three candidates per member tune capacity/regularization by inner-OOF RMSE. Ridge can use training-fold percentile clipping to control extreme input values. Every candidate is fit only inside the available training partition. The primary comparison is against the same equal ensemble without this adaptive selection.

In [ ]:
print(json.dumps(CANDIDATES, indent=2))

## 3. Two-repeat fully nested experiment

Two seeds × five outer folds × three inner folds. Hyperparameters, stopping tree counts, thresholds, and secondary blend weights are selected without outer validation labels. The primary remains equal-weight by design.

An existing run is accepted only if data hashes, code hashes, package versions, configuration, and aggregate artifact hashes match. Set `FORCE_RETRAIN = True` to retrain even a verified matching run. If results are absent or stale, running all cells reproduces training.

In [ ]:
metadata_path = RESULTS_DIR / "nested_metadata.json"
metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}
if not FORCE_RETRAIN and results_match(config, data, metadata):
    summary = pd.read_csv(RESULTS_DIR / "nested_cv_results.csv")
    print("Loaded matching reference run after data/code/environment/config/artifact checks.")
else:
    summary, metadata = run_experiment(config)
verified_metadata = verify_artifacts()
display(summary.sort_values("mean_oof_qwk", ascending=False).round(4))

## 4. Paired uncertainty and rare-class errors

The same participant indices are bootstrapped across repeats and comparators; repeated predictions are not independent children. These are fixed-prediction CIs, not complete retraining or post-selection intervals. A CI containing zero does not establish a reliable gain.

In [ ]:
deltas = pd.read_csv(RESULTS_DIR / "nested_paired_deltas.csv")
display(deltas.round(4))
for row in deltas.itertuples():
    if not row.interval_excludes_zero:
        print(f"{row.reference}: interval includes zero; do not claim a reliable improvement.")

class_metrics = pd.read_csv(RESULTS_DIR / "nested_class_metrics.csv")
primary_classes = class_metrics.loc[class_metrics.setup.eq(PRIMARY)]
display(primary_classes.groupby("class")[["precision", "recall", "f1-score", "support", "true_positives"]].mean().round(3))
print("Class metrics average repeats; true-positive means can be fractional.")
print("The model is not a clinical screening or diagnostic system.")

## 5. Verify the exact exported policy

Submission IDs follow the sample-submission order. The persisted bundle contains the four selected member models, row-local feature contract, equal weights, and training-only calibrated thresholds. Reloaded predictions are checked independently against the saved submission.

In [ ]:
deployment = verified_metadata["deployment"]
print("Exported policy:", deployment["policy"])
print("Weights:", deployment["weights"])
print("Thresholds:", deployment["thresholds"])
print("Selected candidates:", {name: record["candidate"] for name, record in deployment["selection"].items()})
print("Serialization round trip:", deployment["serialization_round_trip_verified"])
submission = pd.read_csv(RESULTS_DIR / "submission_nested.csv")
print("Example test prediction counts:", submission[TARGET].value_counts().sort_index().to_dict())
print("The public example test has no hidden-test/leaderboard evaluation score.")

## 6. Generate the required technical report and anonymous archive

The seven-section `project.pdf` is generated from verified current results, with sources and limitations. The source archive contains sanitized notebooks, code, dependencies, aggregate results, and the report, excluding data, credentials, model binaries, Git history, and personal runtime paths. Perform a final human inspection before manual Moodle upload.

In [ ]:
from src.report import build_report
from src.package import build_archive

build_report()
build_archive()
display(Image(filename=str(RESULTS_DIR / "figures" / "nested_qwk.png")))
display(Image(filename=str(RESULTS_DIR / "figures" / "nested_confusion.png")))